# Statistics for the data

In [2]:
import pandas as pd 
import matplotlib.pyplot as plt
import re
import numpy as np
from datetime import datetime, timedelta
import os
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

Convert data file to .csv

In [ ]:
# Load Excel file
df = pd.read_excel("endo_and_ibs_all_posts.xlsx")
# Save as CSV
df.to_csv("endo_and_ibs_all_posts.csv", index=False)
# Load the CSV file
df = pd.read_csv("../../data/unfiltered/endo_and_ibs_all_posts.csv")


In [ ]:
# Check the dataframe structure and columns
print("DataFrame columns:")
print(df.columns.tolist())
print(f"\nDataFrame shape: {df.shape}")

Group posts by author

In [ ]:
# Count total posts by class/label
posts_by_class = df['label'].value_counts().sort_index()
print("Total posts by class:")
for label, count in posts_by_class.items():
    print(f"Class {label}: {count:,} posts")

print(f"Total posts across all classes: {posts_by_class.sum():,}")

# Percentage breakdown
print("Percentage breakdown of posts:")
for label, count in posts_by_class.items():
    percentage = (count / posts_by_class.sum()) * 100
    print(f"Class {label}: {percentage:.1f}%")

#Count posts per user
posts_per_user = df.groupby("author").size()
print("Posts per user description")
print(posts_per_user.describe())

# Log-plot
plt.figure()
plt.hist(posts_per_user, bins=30)
plt.yscale("log")
plt.xlabel("Number of posts per user")
plt.ylabel("Number of users (log scale)")
plt.title("Distribution of posts per user (log scale)")
plt.show()


Group by class

In [ ]:
# Group users by class and percentage breakdown
total_users_count = df.groupby('label')['author'].nunique()
print(f"Total unique users across all classes: {total_users_count.sum():,}")
for label, count in total_users_count.items():
    print(f"Class {label}: {count:,} users, {count /total_users_count.sum():.1%} of total users")

# Median and average posts per user
post_median = posts_per_user.median()
post_average = posts_per_user.mean()

# Group posts by user and class
posts_per_user_class = df.groupby(['label', 'author']).size().reset_index(name='post_count')
print("Posts per user by class:")
comparison = posts_per_user_class.groupby('label')['post_count'].describe()
print(comparison)

# Overlapping histograms for comparison
plt.figure(figsize=(10, 6))
for label_num in sorted(posts_per_user_class['label'].unique()):
    label_data = posts_per_user_class[posts_per_user_class['label'] == label_num]['post_count']
    plt.hist(label_data, bins=30, alpha=0.7, label=f'Class {label_num}')

plt.yscale("log")
plt.xlabel("Number of posts per user")
plt.ylabel("Number of users (log scale)")
plt.axvline(x = post_median, color='red', linestyle='--', label=f'Median: {post_median:.0f}')
plt.axvline(x = post_average, color='blue', linestyle='--', label=f'Average: {post_average:.0f}')
plt.title("Distribution of posts per user by class (log scale)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Average post length by class

In [ ]:
# Calculate post length in characters for each post
df['text_length'] = df['text'].astype(str).str.len()

# Compute average post length by class
print("Average post length in characters by class:")
avg_length_by_class = df.groupby('label')['text_length'].agg(['mean', 'median', 'std', 'min', 'max', 'count'])

for label in sorted(df['label'].unique()):
    condition = "Endometriosis" if label == 1 else "IBS"
    stats = avg_length_by_class.loc[label]
    
    print(f"\n{condition} (Class {label}):")
    print(f"  Posts count: {stats['count']:,}")
    print(f"  Average length: {stats['mean']:,.0f} characters")
    print(f"  Median length: {stats['median']:,.0f} characters")
    print(f"  Standard deviation: {stats['std']:,.0f} characters")
    print(f"  Min length: {stats['min']:,.0f} characters")
    print(f"  Max length: {stats['max']:,.0f} characters")

# Overall statistics
print(f"\nOverall statistics:")
print(f"  Total posts: {len(df):,}")
print(f"  Average length across all posts: {df['text_length'].mean():,.0f} characters")
print(f"  Median length across all posts: {df['text_length'].median():,.0f} characters")
print(f"  Standard deviation across all posts: {df['text_length'].std():,.0f} characters")

# Histogram comparison
for label in sorted(df['label'].unique()):
    condition = "Endometriosis" if label == 1 else "IBS"
    class_data = df[df['label'] == label]['text_length']
    plt.hist(class_data, bins=50, alpha=0.7, label=f'{condition} (Class {label})')

plt.xlabel('Post Length (characters)')
plt.ylabel('Frequency')
plt.title('Distribution of Post Lengths by Class')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()